# 05 - Residual Stock LGD Model

**Completed but Unsold Stock - Australian Bank-Style LGD Framework**

This notebook focuses on residual stock risk after practical completion, where recovery depends on sell-down execution rather than construction completion.

| Step | Description |
|---|---|
| 1 | Generate synthetic residual stock dataset |
| 2 | Build base LGD from sell-down and carrying-cost drivers |
| 3 | Apply stress scenarios for downturn LGD |
| 4 | Produce EAD-weighted LGD outputs |
| 5 | Export interview-ready summary tables |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(47)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal workout tapes are unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this notebook is a portfolio-project proxy baseline and is **not** internally calibrated to real workout history.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`time_to_sale_base`).


## Objective
Build an interview-ready residual stock LGD module for completed but unsold inventory with weighted outputs.

## Drivers
- Unsold unit count
- Absorption speed
- Discount-to-clear
- Holding cost and time to sale

## Logic
- Base/downturn/final LGD with product-appropriate discounting and weighted aggregation

## Downturn
- Slower absorption, larger discount-to-clear, and higher carry/discount rates

## Validation
- Weighted scenario/segment outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Residual Stock Differs from Development Finance

Residual stock is a **post-completion** problem. Key differences vs development finance:

- Construction execution risk (cost-to-complete, completion uncertainty) is largely behind the project.
- Main risk shifts to **sell-down execution**: unsold units, absorption speed, discount-to-clear, holding cost, and time to sale.
- Recovery path is inventory liquidation over time, not fund-to-complete decisioning.


## 1. Generate Synthetic Residual Stock Dataset


In [ ]:
n_cases = 240
rng = np.random.default_rng(47)

segments = ['metro_apartment', 'suburban_apartment', 'townhouse', 'mixed_stock']
segment_weights = [0.38, 0.27, 0.22, 0.13]

params = {
    'metro_apartment': {'unit_price': (620000, 1150000), 'unsold_units': (15, 120), 'absorption': (3.0, 9.0), 'discount': (0.05, 0.18), 'holding': (0.045, 0.075)},
    'suburban_apartment': {'unit_price': (450000, 780000), 'unsold_units': (12, 90), 'absorption': (2.4, 7.4), 'discount': (0.04, 0.16), 'holding': (0.040, 0.070)},
    'townhouse': {'unit_price': (520000, 920000), 'unsold_units': (8, 60), 'absorption': (1.8, 5.8), 'discount': (0.05, 0.15), 'holding': (0.038, 0.065)},
    'mixed_stock': {'unit_price': (500000, 1100000), 'unsold_units': (10, 95), 'absorption': (2.0, 6.8), 'discount': (0.06, 0.20), 'holding': (0.045, 0.080)},
}

rows = []
for i in range(1, n_cases + 1):
    seg = rng.choice(segments, p=segment_weights)
    p = params[seg]

    unsold_units = int(rng.integers(p['unsold_units'][0], p['unsold_units'][1] + 1))
    avg_unit_price = float(rng.uniform(p['unit_price'][0], p['unit_price'][1]))
    gross_stock_value = unsold_units * avg_unit_price

    absorption_speed_units_per_month = float(rng.uniform(p['absorption'][0], p['absorption'][1]))
    base_time = unsold_units / max(absorption_speed_units_per_month, 0.8)
    time_to_sale_months = float(np.clip(base_time + rng.normal(0.0, 1.3), 3.0, 36.0))

    discount_to_clear = float(rng.uniform(p['discount'][0], p['discount'][1]))
    holding_cost_rate_pa = float(rng.uniform(p['holding'][0], p['holding'][1]))
    selling_cost_rate = float(rng.uniform(0.018, 0.035))

    leverage = float(np.clip(rng.normal(0.68, 0.08), 0.45, 0.88))
    ead = gross_stock_value * leverage * rng.uniform(0.98, 1.03)

    contract_rate_proxy = float(np.clip(rng.normal(0.070, 0.012), 0.045, 0.105))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.054, 0.009), 0.035, 0.085))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    rows.append({
        'case_id': f'RS{i:04d}',
        'segment': seg,
        'unsold_units': unsold_units,
        'avg_unit_price': avg_unit_price,
        'gross_stock_value': gross_stock_value,
        'ead': ead,
        'absorption_speed_units_per_month': absorption_speed_units_per_month,
        'time_to_sale_months': time_to_sale_months,
        'discount_to_clear': discount_to_clear,
        'holding_cost_rate_pa': holding_cost_rate_pa,
        'selling_cost_rate': selling_cost_rate,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

residual = pd.DataFrame(rows)

print('Residual stock cases:', residual.shape)
display(residual['segment'].value_counts())
residual.head()



## 2. Base LGD Calculation


In [ ]:
def compute_residual_stock_lgd(df, speed_multiplier=1.0, discount_addon=0.0, holding_addon=0.0, rate_addon=0.0):
    x = df.copy()

    speed = (x['absorption_speed_units_per_month'] * speed_multiplier).clip(lower=0.8)
    time_to_sale = (x['unsold_units'] / speed).clip(lower=2.0, upper=42.0)

    discount_to_clear = (x['discount_to_clear'] + discount_addon).clip(lower=0.00, upper=0.45)
    holding_cost_rate = (x['holding_cost_rate_pa'] + holding_addon).clip(lower=0.02, upper=0.16)
    discount_rate = (x['discount_rate'] + rate_addon).clip(lower=0.03, upper=0.18)

    gross_sale_proceeds = x['gross_stock_value'] * (1 - discount_to_clear)
    holding_cost = x['gross_stock_value'] * holding_cost_rate * (time_to_sale / 12.0)
    selling_cost = x['gross_stock_value'] * x['selling_cost_rate']

    net_sale_before_discount = (gross_sale_proceeds - holding_cost - selling_cost).clip(lower=0.0)
    discount_factor = (1 + discount_rate) ** (time_to_sale / 12.0)
    net_recovery_pv = net_sale_before_discount / discount_factor

    lgd = ((x['ead'] - net_recovery_pv) / x['ead']).clip(lower=0.00, upper=1.00)

    return lgd, time_to_sale, net_recovery_pv

residual['lgd_base'], residual['time_to_sale_base'], residual['net_recovery_pv_base'] = compute_residual_stock_lgd(residual)

print(f"Portfolio EAD-weighted Base LGD: {exposure_weighted_average(residual, 'lgd_base', 'ead'):.2%}")
display(residual[['lgd_base', 'time_to_sale_base', 'net_recovery_pv_base']].describe().round(4))



## 3. Stress Scenarios, Downturn LGD, and Final Overlay

Stress logic: slower absorption, deeper discount-to-clear, higher carrying cost, and higher discount rate.  
A small explicit final overlay is then applied to downturn LGD so cross-product outputs compare a consistent `lgd_final` metric.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'speed_multiplier': 1.00, 'discount_addon': 0.00, 'holding_addon': 0.000, 'rate_addon': 0.000},
    {'scenario': 'mild_stress', 'speed_multiplier': 0.82, 'discount_addon': 0.04, 'holding_addon': 0.010, 'rate_addon': 0.005},
    {'scenario': 'severe_stress', 'speed_multiplier': 0.65, 'discount_addon': 0.09, 'holding_addon': 0.022, 'rate_addon': 0.015},
]

scenario_rows = []
for s in scenario_specs:
    lgd_s, time_s, rec_s = compute_residual_stock_lgd(
        residual,
        speed_multiplier=s['speed_multiplier'],
        discount_addon=s['discount_addon'],
        holding_addon=s['holding_addon'],
        rate_addon=s['rate_addon'],
    )

    scenario_name = s['scenario']
    residual[f'lgd_{scenario_name}'] = lgd_s
    residual[f'time_to_sale_{scenario_name}'] = time_s
    residual[f'net_recovery_pv_{scenario_name}'] = rec_s

    overlay_s = (
        0.010
        + 0.015 * ((time_s - 12.0) / 24.0).clip(0.0, 1.0)
        + 0.020 * (residual['discount_to_clear'] + s['discount_addon']).clip(0.0, 0.45)
    ).clip(0.010, 0.070)
    residual[f'final_overlay_addon_{scenario_name}'] = overlay_s
    residual[f'lgd_final_{scenario_name}'] = (lgd_s + overlay_s).clip(0.0, 1.0)

    scenario_rows.append({
        'scenario': scenario_name,
        'ead_weighted_lgd_base': exposure_weighted_average(residual, f'lgd_{scenario_name}', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(residual, f'lgd_{scenario_name}', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(residual, f'lgd_final_{scenario_name}', 'ead'),
        'mean_time_to_sale_months': float(time_s.mean()),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_time_to_sale_months']
residual['lgd_downturn'] = residual['lgd_severe_stress']
residual['final_overlay_addon'] = residual['final_overlay_addon_severe_stress']
residual['lgd_final'] = residual['lgd_final_severe_stress']

display(scenario_summary.round(4))



## 4. EAD-Weighted LGD Outputs


In [ ]:
segment_rows = []
for seg, grp in residual.groupby('segment', observed=True):
    segment_rows.append({
        'segment': seg,
        'case_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_unsold_units': grp['unsold_units'].mean(),
        'mean_absorption_speed': grp['absorption_speed_units_per_month'].mean(),
        'mean_discount_to_clear': grp['discount_to_clear'].mean(),
        'mean_holding_cost_rate_pa': grp['holding_cost_rate_pa'].mean(),
        'mean_time_to_sale_months': grp['time_to_sale_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values('ead_weighted_lgd_final', ascending=False).reset_index(drop=True)
display(segment_summary.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = segment_summary.set_index('segment')[['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final']]
plot_df.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_title('Residual Stock LGD by Segment (EAD-Weighted Base/Downturn/Final)')
ax.set_ylabel('LGD')
ax.set_xlabel('Segment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'residual_stock_segment_weighted_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'residual_stock_weighted_lgd_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()



## 5. Validation and Exports


In [ ]:
validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': residual['lgd_base'].between(0, 1).all()},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': residual['lgd_downturn'].between(0, 1).all()},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': residual['lgd_final'].between(0, 1).all()},
    {'check_name': 'downturn_not_below_base', 'passed': (residual['lgd_downturn'] >= residual['lgd_base']).all()},
    {'check_name': 'final_not_below_downturn', 'passed': (residual['lgd_final'] >= residual['lgd_downturn']).all()},
])
display(validation_checks)

residual.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_segment_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_scenario_summary.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_validation_checks.csv'), index=False)

print('Saved residual stock outputs:')
print('- residual_stock_loan_level_output.csv')
print('- residual_stock_segment_summary.csv')
print('- residual_stock_scenario_summary.csv')
print('- residual_stock_validation_checks.csv')



## Assumptions and Limitations

- Residual stock data is synthetic and designed for transparent portfolio demonstration.
- Absorption speed, discount-to-clear, holding cost, and time-to-sale are proxy variables in lieu of internal realised workout tapes.
- Stress scenario parameters are pragmatic interview-grade overlays and should be recalibrated using observed stock liquidation history before production use.
